# Session 3 · de-code app building

*Democratizing Coding with AI — Wed 7 Oct*

<nils.holmberg@iko.lu.se>

## Topics

<!-- table of contents: update when a section heading is renamed or moved -->
- 1 What is an app?
    - 1.1 A flexible definition
    - 1.2 A toy app prompt
    - 1.3 The generated app
- 2 Spec for a Shiny data app
    - 2.1 How a spec works
    - 2.2 Purpose and users
    - 2.3 Data
    - 2.4 Features and layout
    - 2.5 Behaviour
    - 2.6 Technical constraints
    - 2.7 Acceptance checks
    - 2.8 Putting it online
    - 2.9 Save the spec
- 3 Spec for a local note-taking web app
    - 3.1 Why a local web app
    - 3.2 Purpose and users
    - 3.3 Files and constraints
    - 3.4 Data model and storage
    - 3.5 Features and layout
    - 3.6 Look and accessibility
    - 3.7 Acceptance checks
    - 3.8 How to run
    - 3.9 Save the spec
    - 3.10 Your turn
- [Slides](https://de-code-ai.netlify.app/coding/skill-school/sessions/03-app-building-slides.html)
- [Notebook](https://colab.research.google.com/github/nils-holmberg/de-code/blob/main/jnb/skill-school/03-app-building.ipynb)

In [ ]:
#| echo: false
import matplotlib.pyplot as plt

parts = ["layout", "inputs", "logic", "charts", "deploy"]
effort = [3, 4, 6, 5, 2]

fig, ax = plt.subplots()
ax.barh(parts, effort, color="#2f9e44", height=0.62)
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e6e6e6", linewidth=0.8)
ax.set_axisbelow(True)
ax.set_xlabel("relative effort")
ax.set_title("Placeholder: effort per app component")
plt.tight_layout()
plt.show()

# 1 What is an app?

## 1.1 A flexible definition

In this course, an **app** is any program that takes some **input**, does a **job** for
someone, and produces **output** they can use — without them having to read or change the
code. That makes "app" a wide term. What differs is how the input arrives, where the program
runs, and who uses it:

| App | Input | Job | Output | Who runs it |
|---|---|---|---|---|
| Notebook cell with settings at the top | a few variables you edit | analyse a dataset | tables, charts, a report file | you, in Colab |
| Command-line script | a file name, options | convert or clean files | new files | you, in a terminal |
| Spreadsheet with formulas | numbers typed into cells | calculate | totals, charts | anyone with Excel |
| Interactive dashboard (Shiny) | menus, sliders | filter and summarise data | live charts and tables | anyone with the link |
| Local web page (HTML + JavaScript) | clicks, typed text | store and show information | an updated page | anyone who opens the file |
| Hosted web app | forms, accounts | a service | pages, emails, downloads | the public |
| Chatbot (custom GPT, Gemini Gem) | questions in plain language | answer with set instructions | text, files | anyone with access |
| Mobile app | taps, camera, location | everything above, on a phone | screens, notifications | app-store users |

We start at the top of the table, with an app that is a single notebook cell, and work our way down.

## 1.2 A toy app prompt

This prompt asks Gemini for a tiny *report app*: one Colab cell with its settings at the top.
Paste the prompt into [Gemini](https://gemini.google.com/), then paste the code it returns
into a new cell in Colab. The cell below only stores the text.

In [ ]:
#| eval: false
prompt = """
Write one Google Colab Python code cell that works as a small report app.

Settings at the top of the cell, as plain variables the user can edit:
- DATA_URL = "https://raw.githubusercontent.com/nils-holmberg/de-code/main/csv/komp014-simulated-data-n100.tsv"
- OUTCOME = "DV_INDEX", GROUP = "IV_CAT_GROUP", PREFIX = "app"

The cell should:
1. load the tab-separated file from DATA_URL with pandas
2. save a summary table (count, mean, SD, min, quartiles, max) of all columns whose
   name contains INDEX to PREFIX-summary.csv
3. save one histogram per index as PREFIX-hist-<index>.png
4. save a box plot of OUTCOME by GROUP as PREFIX-groups.png
5. save one self-contained HTML report, PREFIX-report.html, with the summary table and
   all charts embedded as base64 images, so it is a single file that opens anywhere
6. print the name and size of every file it wrote

Write all files to the current working folder. Use only pandas and matplotlib. Return
only the code, with short comments.
"""

## 1.3 The generated app

A version of the code the model returns. Run it in the
[session notebook in Colab](https://colab.research.google.com/github/nils-holmberg/de-code/blob/main/jnb/skill-school/03-app-building.ipynb),
then open the **Files** panel (the folder icon on the left) to see the six new files.
Right-click a file to download it; open `app-report.html` in your browser.

In [ ]:
# ---- Settings: change these and run the cell again ----
DATA_URL = "https://raw.githubusercontent.com/nils-holmberg/de-code/main/csv/komp014-simulated-data-n100.tsv"
OUTCOME = "DV_INDEX"
GROUP = "IV_CAT_GROUP"
PREFIX = "app"

import base64
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# 1. Load the data
df = pd.read_csv(DATA_URL, sep="\t")
indices = [c for c in df.columns if "INDEX" in c]
written = []


def save_figure(fig, name):
    """Save a figure as PNG and return it base64-encoded for the HTML report."""
    path = Path(f"{PREFIX}-{name}.png")
    fig.savefig(path, dpi=110, bbox_inches="tight")
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    written.append(path)
    return base64.b64encode(buffer.getvalue()).decode()


# 2. Summary table of the indices
summary = df[indices].describe().T.round(2)
summary_path = Path(f"{PREFIX}-summary.csv")
summary.to_csv(summary_path)
written.append(summary_path)

# 3. One histogram per index
images = {}
for col in indices:
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.hist(df[col], bins=9, color="#2f9e44", edgecolor="white")
    ax.set_title(col)
    images[f"Distribution of {col}"] = save_figure(fig, f"hist-{col.lower()}")

# 4. Outcome by group
fig, ax = plt.subplots(figsize=(4, 3))
df.boxplot(column=OUTCOME, by=GROUP, ax=ax, grid=False)
ax.set_title(f"{OUTCOME} by {GROUP}")
fig.suptitle("")
images[f"{OUTCOME} by {GROUP}"] = save_figure(fig, "groups")

# 5. One self-contained HTML report with the table and all charts embedded
figures = "\n".join(
    f'<h2>{title}</h2><img src="data:image/png;base64,{data}" alt="{title}">'
    for title, data in images.items()
)
html = f"""<!doctype html>
<html><head><meta charset="utf-8"><title>Report: {OUTCOME}</title>
<style>body {{ font-family: Arial, sans-serif; max-width: 800px; margin: 2em auto; }}
table {{ border-collapse: collapse; }} td, th {{ padding: 4px 8px; border: 1px solid #ccc; }}</style>
</head><body>
<h1>Report: {OUTCOME}</h1>
<p>{len(df)} rows, {df.shape[1]} columns, source: {DATA_URL}</p>
<h2>Summary of the indices</h2>
{summary.to_html()}
{figures}
</body></html>"""
report_path = Path(f"{PREFIX}-report.html")
report_path.write_text(html, encoding="utf-8")
written.append(report_path)

# 6. Tell the user what was made
print(f"Wrote {len(written)} files:")
for path in written:
    print(f"  {path.name:28} {path.stat().st_size:>8,} bytes")
summary

Changing a setting at the top — say `OUTCOME = "IV_INDEX1"` or `PREFIX = "run2"` — and
running the cell again *is* using the app: same job, new input, new output, no code reading
required.

# 2 Spec for a Shiny data app

## 2.1 How a spec works

A prompt is a one-off request. A **specification** (spec) is a document you keep: a
plain-text file that describes *what* an app should do and *how to check it*, written so
that both people and AI can follow it. Writing specs as files — *specifications as code* —
lets them do three jobs at once:

- **Context:** the AI reads the spec before it writes anything, so it knows the data, the
  users and the constraints instead of guessing.
- **Instructions:** prompts can stay short — "implement section 3 of SPEC.md" — because the
  details live in the spec.
- **Blueprint:** an AI agent such as [Google Antigravity](https://antigravity.google/) can
  plan the work from the spec, build the app step by step, and test it against the
  acceptance checks.

The workflow: run the spec cells below in Colab to create `SPEC.md`, download it, put it in
an empty folder, open that folder in Antigravity, and ask the agent: *"Read SPEC.md and
build the app step by step. Check every acceptance criterion and report the result."*
When something is wrong, change the spec and ask again — the spec stays the single source
of truth. The cells below only store text; on this page nothing is run.

## 2.2 Purpose and users

Start with *why* and *for whom*. One clear sentence of purpose keeps the agent from adding
features nobody asked for.

In [ ]:
#| eval: false
spec = """
# SPEC: TPB explorer (Shiny for Python)

## 1. Purpose and users

Build an interactive web app that lets students explore a synthetic Theory of Planned
Behavior (TPB) survey experiment: how attitude (ATT), subjective norm (SN) and perceived
behavioral control (PBC) relate to children's intention (BI) to learn an instrument, and
whether a peer testimonial (treatment) raised that intention.

Users: course participants with no coding background. They use the app in a browser,
mainly on a laptop, to look at distributions, relationships and the experiment result.
"""

## 2.3 Data

Describe the data precisely: where it is, every column and its role. The agent then writes
correct loading code on the first try, and the checks at the end tell it whether it did.

In [ ]:
#| eval: false
spec += """
## 2. Data

Load this tab-separated file at start-up (100 rows, no missing values):
https://raw.githubusercontent.com/nils-holmberg/de-code/main/csv/komp014-tpb-exp-n100.tsv

Columns:
- Background (text): age_group ("10 years" to "13 years"), grade_level ("Grade 4" to
  "Grade 6"), gender ("Girl", "Boy", "Non-binary", "Prefer not to say")
- Experiment (text): experimental_condition ("control" or "treatment")
- Items (whole numbers 1-5, 1 = strongly disagree, 5 = strongly agree), three per scale:
  - ATT: att_fun_exciting, att_valuable_use_of_time, att_feel_good_about_self
  - SN: sn_parent_guardian_approval, sn_friend_approval, sn_teacher_encouragement
  - PBC: pbc_confident_can_learn, pbc_enough_spare_time, pbc_instrument_access
  - BI: bi_plan_next_6_months, bi_try_best_to_learn_soon, bi_want_to_start_school_year
- Indices (decimals 1-5): ATT_index, SN_index, PBC_index, BI_index = mean of the three
  items of each scale. Recompute them from the items after loading.

If loading fails or the file does not have 100 rows, show a clear error message in the
app instead of crashing.
"""

## 2.4 Features and layout

Describe what the user sees and can do, screen by screen. Numbered requirements give the
agent a to-do list and give you something concrete to check.

In [ ]:
#| eval: false
spec += """
## 3. Features and layout

Page title "TPB explorer". A sidebar on the left, tabs on the right.

Sidebar filters (all selected by default, apply to every tab):
- F1 experimental condition (checkboxes)
- F2 gender, F3 grade level, F4 age group (checkbox groups)
- F5 a line showing "n = <number of rows after filtering>"
- F6 a button that downloads the filtered rows as CSV

Tabs:
- T1 Overview: table of mean and SD of the four indices, overall and by condition
- T2 Distributions: dropdown to pick one index; histogram with a density curve
- T3 Relationships: dropdowns for x and y index (default PBC_index and BI_index);
  scatter plot with a fitted line; Pearson r and p shown above the plot
- T4 Experiment: box plot of BI_index by condition with the individual points;
  Welch t-test (t, p) and Cohen's d for treatment vs control shown in a small table
"""

## 2.5 Behaviour

Say how the app should *behave*, not just what it shows: what reacts to what, and what
happens in edge cases such as an empty selection.

In [ ]:
#| eval: false
spec += """
## 4. Behaviour

- Every table, plot and number updates immediately when a filter changes.
- If the filters leave fewer than 5 rows, show "Too few rows for this view" instead of
  plots and tests.
- If only one condition is selected, T4 shows "Select both conditions to compare"
  instead of the test.
- Round numbers to 2 decimals; show p-values below .001 as "< .001".
- Use one colour palette across all plots; label every axis with the full construct
  name (for example "Behavioral intention (BI)").
"""

## 2.6 Technical constraints

Constraints decide *how* it is built. Fixing the framework, files and libraries keeps the
result small, runnable and easy to put online.

In [ ]:
#| eval: false
spec += """
## 5. Technical constraints

- Shiny for Python, core API (from shiny import App, ui, render, reactive).
- Libraries: shiny, pandas, matplotlib, scipy. Nothing else.
- Files: app.py (the whole app), requirements.txt (pinned versions), README.md (how to
  run locally and how to deploy, see section 7).
- No database, no login, no files written at run time.
- Runs locally with:  shiny run app.py  and opens at http://127.0.0.1:8000
"""

## 2.7 Acceptance checks

The checks are the *definition of done*. Each one is a statement that is either true or
false, so the agent can test its own work — and so can you.

In [ ]:
#| eval: false
spec += """
## 6. Acceptance checks

- A1 With all filters selected, F5 shows n = 100.
- A2 Unticking "treatment" shows n = 50, and T4 shows "Select both conditions to
  compare" instead of the test.
- A3 The means in T1 equal pandas df.groupby("experimental_condition")[indices].mean()
  rounded to 2 decimals.
- A4 Changing the T3 dropdowns updates the plot and r without reloading the page.
- A5 The downloaded CSV has the same rows as F5 reports.
- A6 shiny run app.py starts without errors or warnings in a fresh virtual
  environment created from requirements.txt.
"""

## 2.8 Putting it online

A spec can also describe how the app goes live, so the agent prepares the right files.
Three realistic routes, from easiest to most independent:

- **Posit Connect Cloud** publishes straight from a GitHub repository and updates the app
  when you push changes.
- **shinyapps.io** uploads from your computer with a command-line tool.
- **Shinylive** turns the app into a static website that runs entirely in the visitor's
  browser — it can be hosted on Netlify or GitHub Pages, like this course site.

In [ ]:
#| eval: false
spec += """
## 7. Putting it online

Write these steps into README.md and make sure the files support them.

Option A - Posit Connect Cloud (recommended):
1. Create a public GitHub repository with app.py, requirements.txt and README.md.
2. Sign in at https://connect.posit.cloud with the GitHub account.
3. Choose Publish, then Shiny, pick the repository, the branch (main) and app.py as
   the primary file, and publish.
4. Share the public URL; every push to main republishes the app.

Option B - shinyapps.io:
1. Create an account at https://www.shinyapps.io and copy a token from the dashboard.
2. In a terminal:
       pip install rsconnect-python
       rsconnect add --account <account> --name <account> --token <token> --secret <secret>
       rsconnect deploy shiny . --name <account> --title tpb-explorer

Option C - Shinylive (static site, no server):
1. In a terminal:
       pip install shinylive
       shinylive export . site
2. Upload the site folder to Netlify or GitHub Pages. The app then runs in the
   visitor's browser; all libraries must be available in Pyodide (shiny, pandas,
   matplotlib and scipy are).

Never put passwords or tokens in the repository. The data are synthetic, so the app can
be public.
"""

## 2.9 Save the spec

The last cell joins all parts into one file. In Colab, run all spec cells, then download
`SPEC.md` from the Files panel and give it to Antigravity.

In [ ]:
#| eval: false
with open("SPEC.md", "w") as f:
    f.write(spec)
print(spec)

# 3 Spec for a local note-taking web app

## 3.1 Why a local web app

Not every app needs a server. A web browser can run an app made of three plain files —
HTML for the structure, CSS for the look, and JavaScript for the behaviour. You open
`index.html` by double-clicking it, and the app runs on your own computer: no install, no
account, no internet connection. Compared with the Shiny app there is no data file to
load; instead the app *creates* data — your notes — and stores it in the browser.

The spec below follows the same pattern as before: purpose, constraints, data, features,
checks. Run its cells in Colab to create `NOTES-SPEC.md` for Antigravity.

## 3.2 Purpose and users

In [ ]:
#| eval: false
notes_spec = """
# SPEC: Course notes (local web app)

## 1. Purpose and users

A simple note-taking app for course participants to write down ideas, prompts and
results during the sessions. One user, one computer, no account.
"""

## 3.3 Files and constraints

These constraints are what make the app *local*: no frameworks to install, nothing loaded
from the internet, and it must work when opened straight from a folder.

In [ ]:
#| eval: false
notes_spec += """
## 2. Files and constraints

- Exactly three files in one folder: index.html, style.css, app.js.
- Plain HTML, CSS and JavaScript. No frameworks, no libraries, no CDN links, no build
  step, no server.
- Must work when index.html is opened directly from the folder (a file:// address),
  also without an internet connection, in current Chrome, Firefox, Edge and Safari.
"""

## 3.4 Data model and storage

A data model says exactly what one note *is*. Storage says where notes are kept — here
`localStorage`, a small database that every browser has for each page.

In [ ]:
#| eval: false
notes_spec += """
## 3. Data model and storage

- A note has: id (unique text), title (text), body (text), tags (list of words),
  created and updated (date and time).
- All notes are saved in the browser's localStorage under the key "course-notes" as
  JSON, after every change.
- Export: a button that downloads all notes as course-notes-<date>.json.
- Import: a button that reads such a file and adds its notes, skipping ids that already
  exist.
"""

## 3.5 Features and layout

In [ ]:
#| eval: false
notes_spec += """
## 4. Features and layout

- N1 Two columns: a list of notes on the left, the selected note in an editor on the
  right. On narrow screens (below 700 px), show one column at a time.
- N2 "New note" button; the new note opens in the editor with the title selected.
- N3 Edit title, body and tags (comma-separated); changes save automatically.
- N4 Delete a note after a confirmation question.
- N5 Search box that filters the list by title and body as you type.
- N6 Click a tag to show only notes with that tag; click again to show all.
- N7 The list is sorted by last updated, newest first, and shows title, date and the
  first line of the body.
- N8 Keyboard shortcuts: Ctrl/Cmd+N new note, Ctrl/Cmd+F focus search, Esc clears the
  search.
"""

## 3.6 Look and accessibility

Saying how the app should look and who must be able to use it avoids an app that only
works for its builder.

In [ ]:
#| eval: false
notes_spec += """
## 5. Look and accessibility

- Clean, readable design: system font, comfortable line length, enough contrast.
- Follow the computer's light or dark mode (prefers-color-scheme).
- Every control has a visible label or an accessible name; everything works with the
  keyboard, with a visible focus outline.
"""

## 3.7 Acceptance checks

In [ ]:
#| eval: false
notes_spec += """
## 6. Acceptance checks

- A1 Opening index.html from the folder shows the app with no errors in the browser
  console.
- A2 A new note with a title, body and two tags is still there after closing and
  reopening the browser.
- A3 Searching for a word in a body shows only notes containing that word.
- A4 Export, delete all notes, import the exported file: every note is back.
- A5 The app works with the computer offline.
- A6 At 400 px width, all features are reachable.
"""

## 3.8 How to run

In [ ]:
#| eval: false
notes_spec += """
## 7. How to run

Write this into a short README.md:
1. Keep index.html, style.css and app.js together in one folder.
2. Double-click index.html, or drag it into a browser window.
3. Notes are stored in that browser on that computer only. Use Export regularly as a
   backup, and Import to move notes to another browser or computer.
"""

## 3.9 Save the spec

In [ ]:
#| eval: false
with open("NOTES-SPEC.md", "w") as f:
    f.write(notes_spec)
print(notes_spec)

## 3.10 Your turn

Pick an app idea of your own — a reading list, a habit tracker, a quiz for your students,
a dashboard for your own data. Copy one of the specs, then:

1. Rewrite **Purpose and users** for your idea.
2. Replace **Data** (or the data model) and **Features** with what your app needs.
3. Keep the **constraints** that make it easy to run, and write **acceptance checks** you
   could test yourself.
4. Give the spec to Antigravity, check the result against your acceptance checks, and
   improve the spec — not the code — until it passes.